# Preprocessing

In [9]:
import pandas as pd
import os

# Configuration -
CONFIG = {
    "input_folder": "01_Raw Data",
    "input_file": "Stromerzeugung.xlsx",    
    "output_folder": "02_Preprocessing",
    "csv_filename": "Gesamterzeugung_hourly.csv"
}

#os.chdir(os.path.dirname(os.path.abspath(__file__)))
#input_path = 
df = pd.read_excel(os.path.join(CONFIG['input_folder'], CONFIG['input_file']))

# Excel file in the same folder as the notebook
#df = pd.read_excel(f"{CONFIG['input_folder']}/{CONFIG['input_file']}")
#df = pd.read_excel('Stromerzeugung.xlsx')
print(f"Loaded: {df.shape}")



Loaded: (364412, 13)


In [11]:
# Remove [MWh] from all column names
df.columns = df.columns.str.replace(' \[MWh\]', '', regex=True).str.strip()

columns = list(range(len(df.columns)))# all column indices from df
#columns = [0,1,2,3]

# Checking classes  of the columns
for col_idx in columns:
    col_name = df.columns[col_idx]
    print(f"Column {col_name:<40} {df[col_name].apply(type).unique()}")
   





Column Datum von                                [<class 'str'>]
Column Biomasse                                 [<class 'float'> <class 'int'> <class 'str'>]
Column Wasserkraft                              [<class 'float'> <class 'int'> <class 'str'>]
Column Wind Offshore                            [<class 'int'> <class 'float'> <class 'str'>]
Column Wind Onshore                             [<class 'float'>]
Column Photovoltaik                             [<class 'float'>]
Column Sonstige Erneuerbare                     [<class 'float'> <class 'int'> <class 'str'>]
Column Kernenergie                              [<class 'float'>]
Column Braunkohle                               [<class 'float'>]
Column Steinkohle                               [<class 'float'> <class 'int'> <class 'str'>]
Column Erdgas                                   [<class 'float'> <class 'int'> <class 'str'>]
Column Pumpspeicher                             [<class 'int'> <class 'float'> <class 'str'>]
Column Sonstig

In [12]:
# Convert columns to the correct format
columns.remove(0)

for col_idx in columns:
    if col_idx < len(df.columns):
        col_name = df.columns[col_idx]
        #print(f"Fixing: {col_name}")
        df[col_name] = pd.to_numeric(df[col_name], errors='coerce')



print("All columns are now numeric!")


All columns are now numeric!


In [13]:
display(df)

,Datum von,Biomasse,Wasserkraft,Wind Offshore,Wind Onshore,Photovoltaik,Sonstige Erneuerbare,Kernenergie,Braunkohle,Steinkohle,Erdgas,Pumpspeicher,Sonstige Konventionelle
0,01.01.2015 00:00,1005.50,288.25,130.00,2028.25,0.0,33.25,2685.50,3964.75,805.50,317.25,391.00,1235.00
1,01.01.2015 00:15,1007.00,287.75,129.25,2023.00,0.0,33.25,2646.25,3950.75,833.00,316.25,373.25,1213.75
2,01.01.2015 00:30,1006.50,292.75,128.50,2040.25,0.0,33.25,2660.75,3912.25,806.25,313.50,426.50,1218.50
3,01.01.2015 00:45,1005.25,289.50,128.75,2036.50,0.0,33.25,2718.00,3859.50,775.00,279.25,335.00,1242.00
4,01.01.2015 01:00,999.00,295.25,128.75,2045.75,0.0,33.25,2772.25,3888.00,633.50,261.50,247.50,1247.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...
364407,23.05.2025 22:45,1008.50,426.75,948.75,3108.25,0.0,23.25,0.00,1810.75,530.75,1443.00,805.75,441.75
364408,23.05.2025 23:00,1007.75,443.75,927.25,3092.25,0.0,23.25,0.00,1735.50,508.75,1408.50,994.75,442.75
364409,23.05.2025 23:15,1007.75,428.25,894.25,3120.00,0.0,23.25,0.00,1717.25,471.00,1275.25,816.25,443.50
364410,23.05.2025 23:30,1009.25,415.50,879.00,3172.50,0.0,23.00,0.00,1747.25,442.25,1212.75,649.25,443.00


In [14]:
#New column 'Stromerzeugung Gesamt' as sum of columns 1 to 12
df['Stromerzeugung Gesamt'] = df.iloc[:, 1:13].sum(axis=1)
# Rename first column to 'Datum'
df.rename(columns={df.columns[0]: 'Datum'}, inplace=True)

# Convert 'Datum' column to datetime with day first format
df['Datum'] = pd.to_datetime(df['Datum'], dayfirst=True)

neu = df[['Datum', df.columns[13]]].copy()
neu = neu.set_index('Datum')

neu = neu.resample('h').sum()
display(neu)


,Stromerzeugung Gesamt
Datum,
2015-01-01 00:00:00,51238.75
2015-01-01 01:00:00,49749.00
2015-01-01 02:00:00,49014.75
2015-01-01 03:00:00,47954.75
2015-01-01 04:00:00,48187.00
...,...
2025-05-23 19:00:00,49496.25
2025-05-23 20:00:00,45523.75
2025-05-23 21:00:00,42886.00


In [ ]:
#Save
neu.to_csv(os.path.join(CONFIG['output_folder'], CONFIG['csv_filename']), index=True)

In [16]:
# Checking if it was saved correctly
neu2 = pd.read_csv(os.path.join(CONFIG['output_folder'], CONFIG['csv_filename']), index_col='Datum', parse_dates=True)

display(neu2)

,Stromerzeugung Gesamt
Datum,
2015-01-01 00:00:00,51238.75
2015-01-01 01:00:00,49749.00
2015-01-01 02:00:00,49014.75
2015-01-01 03:00:00,47954.75
2015-01-01 04:00:00,48187.00
...,...
2025-05-23 19:00:00,49496.25
2025-05-23 20:00:00,45523.75
2025-05-23 21:00:00,42886.00
